# Binding Pocket Detection

In [1]:
%load_ext autoreload
%autoreload 2

import time
from pathlib import Path
import pickle


import numpy as np
from pdbfixer import PDBFixer
from openmm.app import PDBFile
import caddpy
import sciapi
import scifile
import scishow

In [2]:
project_id = "3w32"
pdb_id = project_id
cache_dir = Path(f".tmp/{project_id}")
cache_dir.mkdir(exist_ok=True, parents=True)
pdb_filepath_raw = cache_dir / "receptor_raw.pdb"
pdb_filepath_final = cache_dir / "receptor_fixed_apo.pdb"
dogsite_pockets_filepath = cache_dir / "pockets.pkl"
detector_filepath = cache_dir / "detector.pkl"

In [3]:
if not pdb_filepath_raw.is_file():
    pdb_file_content = sciapi.pdb.file.entry(pdb_id=pdb_id, file_format="pdb")
    pdb_filepath_raw.write_bytes(pdb_file_content)

In [4]:
if not pdb_filepath_final.is_file():
    fixer = PDBFixer(filename=str(pdb_filepath_raw))
    fixer.removeHeterogens(keepWater=False)
    fixer.addMissingHydrogens(7.0)
    PDBFile.writeFile(fixer.topology, fixer.positions, open(pdb_filepath_final, 'w'))

In [5]:
receptor = caddpy.chemsys.from_pdb(pdb_filepath_final)

In [12]:
if not detector_filepath.is_file():
    t_start=time.time()
    detector = caddpy.pocket.detector(receptor, gui=True, display=False, grid=0.4)
    t_end=time.time()
    print("Calculation time:", t_end - t_start)
    with open(detector_filepath, "wb") as f:
        pickle.dump(detector._detector, f)
else:
    with open(detector_filepath, "rb") as f:
        detector = caddpy.pocket.DetectorGUI(detector=pickle.load(f))

In [ ]:
if not dogsite_pockets_filepath.is_file():
    protplus_upload_results = sciapi.proteinsplus().upload_pdb(receptor.to_pdb().to_file().encode())
    dogsite_results = sciapi.proteinsplus().dogsite(
        pdb_id=protplus_upload_results.dummy_pdb_id,
        algorithm="scorer",
    )
    dogsites = dogsite_results.full_data
    with open(dogsite_pockets_filepath, "wb") as f:
        pickle.dump(dogsites, f)
else:
    with open(dogsite_pockets_filepath, "rb") as f:
        dogsites = pickle.load(f)

for dogsite in dogsites:
    dogsite_pocket = scifile.mrc.read(dogsite.pop("mrc"))
    detector.nglwidget.add_volume(
        dogsite_pocket.data,
        basis=dogsite_pocket.grid_vectors,
        origin=dogsite_pocket.grid_origin,
        name=f"DoG_{dogsite["name"].removeprefix("P")}",
        representation_params=scishow.nglview.SurfaceRepresentationParameters(
            lazy=True, opacity=0.8, contour=True, visible=True, color=(30,30,30), isolevel=1, isolevel_type="value"
        )
    )

In [13]:
detector.display()

NGLWidget(gui_style='ngl')

Accordion(children=(Output(),), titles=('Logs',))

In [15]:
detector.pockets.pockets

,label,volume,point_count,is_subpocket,parent_label,pocket
0,1,207.141916,3266,False,1,<caddpy.pocket.pocket.Pocket object at 0x170b2...
1,2,47.440953,748,False,2,<caddpy.pocket.pocket.Pocket object at 0x180d8...
2,3,758.801557,11964,False,3,<caddpy.pocket.pocket.Pocket object at 0x170b6...
3,4,102.619602,1618,False,4,<caddpy.pocket.pocket.Pocket object at 0x170ab...
4,5,67.482853,1064,False,5,<caddpy.pocket.pocket.Pocket object at 0x17138...
5,6,38.181088,602,False,6,<caddpy.pocket.pocket.Pocket object at 0x170ab...
6,7,50.295021,793,False,7,<caddpy.pocket.pocket.Pocket object at 0x171bb...
7,8,48.963123,772,False,8,<caddpy.pocket.pocket.Pocket object at 0x171bb...
8,9,38.434783,606,False,9,<caddpy.pocket.pocket.Pocket object at 0x1713d...


In [ ]:
import scipy as sp

In [ ]:
eroded = sp.ndimage.binary_erosion(detector.mask)
prop = sp.ndimage.binary_propagation(eroded, mask=detector.mask)

In [ ]:
import numpy as np
comparison = detector.mask != prop

In [ ]:
np.count_nonzero(comparison)

In [ ]:
a = np.zeros((10,10), dtype=int)
a[0:3, 0:3] = 1
a[2:5,3] = 1
a[4:7, 4:7] = 1
a

In [ ]:
a_opened = sp.ndimage.binary_opening(a).astype(int)
a_opened

In [ ]:
a_opened_closed = sp.ndimage.binary_closing(a_opened).astype(int)
a_opened_closed

In [ ]:
b = sp.ndimage.binary_erosion(a)
b.astype(int)

In [ ]:
sp.ndimage.binary_propagation(b, mask=a).astype(int)

In [ ]:
x=detector.nglwidget.add_volume(
    data=pocket_labels.astype(bool),
    basis=detector.field.grid.spacings,
    origin=detector.field.grid.lower_bounds,
    name=f"Pok",
    representation_params=scishow.nglview.RepresentationParameters(opacity=0.7, visible=False, lazy=True)
    )

In [ ]:
import matplotlib.pyplot as plt

def show(array1: np.ndarray, array2: np.ndarray, dpi=None) -> None:
    """Plot differences between two boolean 2D numpy arrays.

    Highlights pixels that are on in both arrays (black), removed pixels (red),
    and added pixels (green). Pixels off in both arrays are shown in white.

    Parameters
    ----------
    array1 : np.ndarray
        First boolean 2D array.
    array2 : np.ndarray
        Second boolean 2D array.

    Raises
    ------
    ValueError
        If the input arrays do not have the same shape or are not boolean 2D arrays.

    Notes
    -----
    Requires matplotlib.
    """
    # Validate inputs
    if array1.shape != array2.shape:
        raise ValueError(f"Input arrays must have the same shape, got {array1.shape} and {array2.shape}.")
    if array1.ndim != 2 or array2.ndim != 2:
        raise ValueError("Input arrays must be 2D.")
    if not (np.issubdtype(array1.dtype, np.bool_) and np.issubdtype(array2.dtype, np.bool_)):
        raise ValueError("Input arrays must be of boolean dtype.")

    # Initialize an image with white background
    height, width = array1.shape
    image = np.ones((height, width, 3), dtype=float)

    # Both arrays true -> black
    both_mask = array1 & array2
    image[both_mask] = [0.2, 0.2, 0.2]

    # True in array1, False in array2 -> red (removed)
    removed_mask = array1 & ~array2
    image[removed_mask] = [1.0, 0.0, 0.0]

    # False in array1, True in array2 -> green (added)
    added_mask = ~array1 & array2
    image[added_mask] = [0.0, 1.0, 0.0]

    # Display the result
    fig = plt.figure(figsize=(800, 800*height/width), dpi=dpi or int(min(width, height)/50))
    fig.canvas.header_visible = False   # hides “Figure X” bar
    fig.canvas.toolbar_visible = False  # hides the floating toolbar
    fig.canvas.footer_visible = False   # hides the (x,y) readout bar
    ax = fig.add_axes([0,0,1,1])
    ax.imshow(image, interpolation='nearest')
    # ax.axis('off')
    plt.show()


In [ ]:
v=detector.field.tensor.astype(bool)
section = v[50]

In [ ]:
from scipy import ndimage
fill_holes = ndimage.binary_fill_holes(
    input=section,
    origin=0
)
show(section, fill_holes, dpi=72)

In [ ]:
structure = None
closing = ndimage.binary_closing(
    input=section,
    structure=structure,
    iterations=1,
    origin=0,
    mask=None,
    border_value=1,
    brute_force=False,
)
show(section, closing)